# Citadel — Phase 1: Train the Commander (GRPO on Qwen2.5-3B)

Train the Commander LLM against a **rule-based Oversight** and a curriculum of adversary generations. The Commander learns to:
1. Contain incidents (Bastion v1 core reward)
2. Write justifications that pass Oversight on the first pass
3. Follow governance pre-requisites
4. Cite relevant playbook lessons

Setup: Colab T4. ~200 GRPO steps. Checkpoint saved to `./checkpoints/commander/`.

In [ ]:
# --- Install ---
!pip install -q trl==0.11.4 transformers==4.45.2 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.0.1
!pip install -q openenv-core fastapi pydantic uvicorn openai
!pip install -q matplotlib

In [ ]:
import os, sys, random, json
from pathlib import Path

# Clone Citadel into the runtime if needed
CITADEL_ROOT = Path('/content/Citadel')
if not CITADEL_ROOT.exists():
    !git clone https://github.com/<you>/<repo>.git /content/repo
    # Or copy your local Citadel/ folder into /content/Citadel
sys.path.insert(0, str(CITADEL_ROOT))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

from environment import CitadelEnvironment
from baseline import oversight_rule_based
from models import IncidentAction, CommanderProposal
from inference import (
    COMMANDER_SYSTEM_PROMPT, format_commander_observation, parse_commander_response
)

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True
)

In [ ]:
# --- Reward function for the Commander ---
# Each "prompt" is a current Commander observation (from env.reset or env.step).
# Each "completion" is the Commander's JSON response.
# The env applies the action (with rule-based Oversight) and returns a reward.

def make_env(task_id, adversary_gen):
    env = CitadelEnvironment(oversight_policy=oversight_rule_based)
    env.reset(task_id=task_id, adversary_gen=adversary_gen)
    return env

def commander_reward_fn(prompts, completions, **kwargs):
    """
    For each (prompt, completion) pair, re-run the env step with the
    Commander's parsed action and return the scalar reward.
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        meta = kwargs.get('env_state', [{}] * len(prompts))
        task_id = meta.get('task_id', 'easy_1') if isinstance(meta, dict) else 'easy_1'
        gen = meta.get('adversary_gen', 1) if isinstance(meta, dict) else 1
        env = make_env(task_id, gen)
        action = parse_commander_response(completion)
        obs = env.step(action)
        rewards.append(float(obs.reward or 0.0))
    return rewards

In [ ]:
# --- Prompt dataset: curriculum-balanced initial observations ---
from datasets import Dataset

def build_prompt_examples(n_per_condition=10):
    examples = []
    for task_id in ['easy_1', 'medium_1', 'hard_1']:
        for gen in [1, 2, 3]:
            for seed in range(n_per_condition):
                env = CitadelEnvironment(oversight_policy=oversight_rule_based)
                obs = env.reset(task_id=task_id, seed=seed, adversary_gen=gen)
                user_msg = format_commander_observation(obs.model_dump(), step=0, history=[])
                prompt = tokenizer.apply_chat_template([
                    {'role': 'system', 'content': COMMANDER_SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_msg},
                ], tokenize=False, add_generation_prompt=True)
                examples.append({'prompt': prompt, 'task_id': task_id, 'adversary_gen': gen, 'seed': seed})
    return Dataset.from_list(examples)

train_ds = build_prompt_examples(n_per_condition=8)
print(f'Training set: {len(train_ds)} prompts')

In [ ]:
# --- GRPO training ---

config = GRPOConfig(
    output_dir='./checkpoints/commander',
    num_train_epochs=1,
    max_steps=200,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,
    max_prompt_length=1800,
    max_completion_length=300,
    temperature=0.7,
    logging_steps=5,
    save_steps=50,
    bf16=True,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=[commander_reward_fn],
    train_dataset=train_ds,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model('./checkpoints/commander/final')

In [ ]:
# --- Reward curve plot ---
import matplotlib.pyplot as plt
log = trainer.state.log_history
rewards = [x.get('reward') for x in log if x.get('reward') is not None]
plt.figure(figsize=(10, 4))
plt.plot(rewards)
plt.title('Commander — Reward over GRPO steps')
plt.xlabel('logging step'); plt.ylabel('reward')
plt.savefig('./checkpoints/commander/reward_curve.png', dpi=120, bbox_inches='tight')
plt.show()